# Results Processing Notebook

This notebook processes the black box sample results returned each week.
It reads input/output text files from `data/results/`, converts them to `.npy` files,
and saves them into the appropriate function data folders.

**Workflow:**
1. Prompt for the week number to process
2. Read the text files containing inputs and outputs for all 8 functions
3. Combine with initial data to create updated `.npy` files
4. Display the data in tabular form
5. Show convergence graphs for each function

### Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Base paths (relative to this notebook's location in functions/results/)
DATA_BASE = '../../data'
RESULTS_DIR = os.path.join(DATA_BASE, 'results')

# Function names and their input dimensions
FUNCTIONS = ['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8']

print('Libraries loaded successfully.')
print(f'Results directory: {os.path.abspath(RESULTS_DIR)}')

### Step 2: Prompt for Week Number

In [ ]:
# Prompt for the week number to process
week_number = int(input('Enter the week number to process: '))
print(f'\nProcessing results for Week {week_number}')

# Construct file paths
inputs_file = os.path.join(RESULTS_DIR, f'inputs - week {week_number}.txt')
outputs_file = os.path.join(RESULTS_DIR, f'outputs - week {week_number}.txt')

# Verify files exist
assert os.path.exists(inputs_file), f'Input file not found: {inputs_file}'
assert os.path.exists(outputs_file), f'Output file not found: {outputs_file}'
print(f'Input file:  {inputs_file}')
print(f'Output file: {outputs_file}')

### Step 3: Parse the Text Files

Each line in the text files represents one weekly submission.
Each line contains 8 arrays (one per function).
We parse these and separate the data per function.

In [ ]:
def group_records(text):
    """
    Split raw text into logical records using bracket-depth grouping.
    Each record starts with '[' and ends when bracket count returns to zero.
    This handles arrays that wrap across multiple physical lines (e.g. f8's 8D arrays).
    """
    records = []
    current = ''
    depth = 0
    
    for char in text:
        if char == '[':
            depth += 1
            current += char
        elif char == ']':
            depth -= 1
            current += char
            if depth == 0:
                records.append(current.strip())
                current = ''
        elif depth > 0:
            current += char
        # Characters outside brackets (newlines between records) are ignored
    
    return records


def parse_inputs_file(filepath):
    """
    Parse the inputs text file.
    Each record contains a list of 8 numpy arrays, one per function.
    Handles multi-line wrapping (e.g. f8's 8D arrays span 2 physical lines).
    Returns a dict: {func_name: np.ndarray of shape (num_submissions, input_dim)}
    """
    with open(filepath, 'r') as f:
        text = f.read()
    
    records = group_records(text)
    func_inputs = {fn: [] for fn in FUNCTIONS}
    
    for rec_num, record in enumerate(records, 1):
        # Replace 'array(' with 'np.array(' so eval works with numpy
        record_eval = record.replace('array(', 'np.array(')
        try:
            arrays = eval(record_eval)
        except Exception as e:
            raise ValueError(
                f'Failed to parse input record {rec_num}: {e}\n'
                f'Record content: {record[:200]}...'
            )
        
        if len(arrays) != len(FUNCTIONS):
            raise ValueError(
                f'Input record {rec_num} has {len(arrays)} arrays, expected {len(FUNCTIONS)}'
            )
        
        for i, fn in enumerate(FUNCTIONS):
            func_inputs[fn].append(arrays[i])
    
    # Convert lists to numpy arrays
    for fn in FUNCTIONS:
        func_inputs[fn] = np.array(func_inputs[fn])
    
    return func_inputs


def parse_outputs_file(filepath):
    """
    Parse the outputs text file.
    Each record contains a list of 8 float values, one per function.
    Uses bracket-depth grouping for robustness.
    Returns a dict: {func_name: np.ndarray of shape (num_submissions,)}
    """
    with open(filepath, 'r') as f:
        text = f.read()
    
    records = group_records(text)
    func_outputs = {fn: [] for fn in FUNCTIONS}
    
    for rec_num, record in enumerate(records, 1):
        try:
            values = eval(record)
        except Exception as e:
            raise ValueError(
                f'Failed to parse output record {rec_num}: {e}\n'
                f'Record content: {record[:200]}...'
            )
        
        if len(values) != len(FUNCTIONS):
            raise ValueError(
                f'Output record {rec_num} has {len(values)} values, expected {len(FUNCTIONS)}'
            )
        
        for i, fn in enumerate(FUNCTIONS):
            func_outputs[fn].append(float(values[i]))
    
    # Convert to numpy arrays
    for fn in FUNCTIONS:
        func_outputs[fn] = np.array(func_outputs[fn])
    
    return func_outputs


# Parse both files
new_inputs = parse_inputs_file(inputs_file)
new_outputs = parse_outputs_file(outputs_file)

# --- Out-of-range input validation (T023) ---
# Warn on values outside valid range [0.0, 1.0]
for fn in FUNCTIONS:
    arr = new_inputs[fn]
    for row_idx in range(arr.shape[0]):
        for col_idx in range(arr.shape[1]):
            val = arr[row_idx, col_idx]
            if val < 0.0 or val > 1.0:
                print(f'⚠️  Out-of-range value in {fn}, submission {row_idx+1}, '
                      f'dim {col_idx+1}: {val:.6f}')

# Display summary of parsed data
print(f'\nParsed submission data:')
print(f'{"Function":<10} {"Submissions":<15} {"Input Dims":<12}')
print('-' * 37)
for fn in FUNCTIONS:
    print(f'{fn:<10} {new_inputs[fn].shape[0]:<15} {new_inputs[fn].shape[1]:<12}')

### Step 4: Combine with Initial Data and Save `.npy` Files

For each function, we concatenate the initial data with all the weekly submissions
to create the updated dataset, then save as `.npy` files.

In [ ]:
updated_inputs = {}
updated_outputs = {}

# Always build the combined data (needed by display and convergence cells)
for fn in FUNCTIONS:
    initial_inp = np.load(os.path.join(DATA_BASE, fn, 'initial_inputs.npy'))
    initial_out = np.load(os.path.join(DATA_BASE, fn, 'initial_outputs.npy'))
    updated_inputs[fn] = np.vstack([initial_inp, new_inputs[fn]])
    updated_outputs[fn] = np.concatenate([initial_out, new_outputs[fn]])

# --- Overwrite guard (FR-014) ---
existing_files = []
for fn in FUNCTIONS:
    inp_path = os.path.join(DATA_BASE, fn, f'updated_inputs - Week {week_number}.npy')
    out_path = os.path.join(DATA_BASE, fn, f'updated_outputs - Week {week_number}.npy')
    if os.path.exists(inp_path):
        existing_files.append(inp_path)
    if os.path.exists(out_path):
        existing_files.append(out_path)

proceed = True
if existing_files:
    print(f'⚠️  WARNING: {len(existing_files)} file(s) for Week {week_number} already exist:')
    for ef in existing_files[:6]:
        print(f'   {ef}')
    if len(existing_files) > 6:
        print(f'   ... and {len(existing_files) - 6} more')
    confirm = input('Overwrite existing files? (yes/no): ').strip().lower()
    proceed = confirm in ('yes', 'y')

if not proceed:
    print('Save skipped — existing files were NOT overwritten.')
else:
    print(f'Saving updated .npy files for Week {week_number}:')
    print(f'{"Function":<10} {"Initial":<10} {"+ New":<10} {"= Total":<10} {"Saved To"}')
    print('-' * 70)

    for fn in FUNCTIONS:
        inp_path = os.path.join(DATA_BASE, fn, f'updated_inputs - Week {week_number}.npy')
        out_path = os.path.join(DATA_BASE, fn, f'updated_outputs - Week {week_number}.npy')
        np.save(inp_path, updated_inputs[fn])
        np.save(out_path, updated_outputs[fn])

        initial_inp = np.load(os.path.join(DATA_BASE, fn, 'initial_inputs.npy'))
        n_initial = len(initial_inp)
        n_new = len(new_inputs[fn])
        n_total = len(updated_inputs[fn])
        print(f'{fn:<10} {n_initial:<10} {n_new:<10} {n_total:<10} data/{fn}/')

    print(f'\nAll .npy files saved successfully for Week {week_number}.')

### Step 5: Display Updated Inputs and Outputs in Tabular Form

Show the complete updated dataset for each function as a table.

In [ ]:
for fn in FUNCTIONS:
    inp = updated_inputs[fn]
    out = updated_outputs[fn]
    n_initial = len(np.load(os.path.join(DATA_BASE, fn, 'initial_inputs.npy')))
    
    # Build a DataFrame with input columns and output
    n_dims = inp.shape[1]
    columns = [f'x{i+1}' for i in range(n_dims)] + ['y (output)']
    data = np.column_stack([inp, out])
    df = pd.DataFrame(data, columns=columns)
    
    # Add a source column to indicate initial vs weekly submission
    source = ['Initial'] * n_initial + [f'Week {i+1}' for i in range(len(inp) - n_initial)]
    df.insert(0, 'Source', source)
    
    print(f'\n{"=" * 80}')
    print(f'Function {fn.upper()} — Updated Data (Week {week_number})')
    print(f'{"=" * 80}')
    display(df)
    print()

### Step 6: Convergence Graphs

For each function, plot the convergence of the best observed output value over time.
This shows whether the Bayesian optimisation is finding better solutions each week.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle(f'Output Convergence — Week {week_number}', fontsize=16, fontweight='bold')

for idx, fn in enumerate(FUNCTIONS):
    ax = axes[idx // 4, idx % 4]
    out = updated_outputs[fn]
    n_initial = len(np.load(os.path.join(DATA_BASE, fn, 'initial_inputs.npy')))

    # Running maximum — all problems are maximisation tasks
    running_max = np.maximum.accumulate(out)

    # Plot all observed values
    sample_indices = np.arange(1, len(out) + 1)
    ax.scatter(sample_indices[:n_initial], out[:n_initial],
               c='steelblue', label='Initial', zorder=3, s=30)
    ax.scatter(sample_indices[n_initial:], out[n_initial:],
               c='orangered', label='BO Submissions', zorder=3, s=50, marker='D')

    # Plot running best (max only — maximisation)
    ax.plot(sample_indices, running_max, 'g-', alpha=0.7, linewidth=2, label='Best Found')

    # Mark the boundary between initial and BO samples
    ax.axvline(x=n_initial + 0.5, color='gray', linestyle=':', alpha=0.5)

    ax.set_title(f'{fn.upper()}', fontsize=13)
    ax.set_xlabel('Sample Number')
    ax.set_ylabel('Output Value')
    ax.legend(fontsize=7, loc='best')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Step 7: Summary Statistics

A quick summary of the best values found for each function after this week's update.

In [ ]:
print(f'Summary — Best Observed Values (Week {week_number})')
print(f'{"Function":<10} {"Min Output":<20} {"Max Output":<20} {"Best Input (at Max)"}')
print('-' * 80)

for fn in FUNCTIONS:
    inp = updated_inputs[fn]
    out = updated_outputs[fn]
    min_idx = np.argmin(out)
    max_idx = np.argmax(out)

    min_val = out[min_idx]
    max_val = out[max_idx]
    best_input = inp[max_idx]

    # Format the best input
    input_str = ', '.join([f'{v:.6f}' for v in best_input])
    print(f'{fn:<10} {min_val:<20.6e} {max_val:<20.6e} [{input_str}]')